In [2]:
import re
from pathlib import Path

# Nếu True: bỏ qua khi dòng đã có {#...} ở BẤT KỲ đâu
# Nếu False: chỉ bỏ qua khi {#...} nằm ở CUỐI dòng
SKIP_IF_ANCHOR_ANYWHERE = False

# Regex nhận diện anchor {#...} (id có thể là số, chữ, gạch ngang, gạch dưới)
ANCHOR_AT_END_RE = re.compile(r'\{#[^}]+\}\s*$')
ANCHOR_ANYWHERE_RE = re.compile(r'\{#[^}]+\}')


def add_anchor_to_line(line: str) -> str:
    """
    Thêm anchor {#number} vào cuối dòng nếu dòng bắt đầu bằng number\.
    Bỏ qua nếu dòng đã có anchor {#...}.
    Ví dụ: '25\. abc...' -> '25\. abc... {#25}'
    """
    # Tách ký tự xuống dòng
    if line.endswith('\n'):
        content = line[:-1]
        newline = '\n'
    else:
        content = line
        newline = ''

    # === KIỂM TRA ĐÃ CÓ ANCHOR CHƯA ===
    if SKIP_IF_ANCHOR_ANYWHERE:
        if ANCHOR_ANYWHERE_RE.search(content):
            return line
    else:
        if ANCHOR_AT_END_RE.search(content):
            return line

    # Tách phần indent (khoảng trắng đầu dòng)
    stripped = content.lstrip()
    indent = content[:len(content) - len(stripped)]

    # Kiểm tra bắt đầu bằng số
    i = 0
    while i < len(stripped) and stripped[i].isdigit():
        i += 1
    if i == 0:
        return line
    number = stripped[:i]
    rest = stripped[i:]

    # Phải bắt đầu bằng `\.` (backslash + dấu chấm)
    if not rest.startswith('\\.'):
        return line

    # Thêm anchor vào cuối dòng (sau khi xóa khoảng trắng thừa)
    content_stripped = content.rstrip()
    new_content = f"{content_stripped} {{#{number}}}"
    return new_content + newline


def process_file(file_path: str, dry_run: bool = False):
    """Đọc file, thêm anchor cho từng dòng, ghi lại file."""
    path = Path(file_path)
    if not path.exists():
        print(f"❌ File không tồn tại: {file_path}")
        return

    with open(path, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    new_lines = [add_anchor_to_line(line) for line in lines]

    # Đếm số dòng thay đổi
    changed = sum(1 for old, new in zip(lines, new_lines) if old != new)

    if dry_run:
        print(f"🔍 [DRY RUN] {file_path}: {changed} dòng sẽ thay đổi")
        # In ra các dòng thay đổi để kiểm tra
        for old, new in zip(lines, new_lines):
            if old != new:
                print(f"   - {old.rstrip()}")
                print(f"   + {new.rstrip()}")
        return

    with open(path, 'w', encoding='utf-8') as f:
        f.writelines(new_lines)

    print(f"✅ Đã xử lý: {file_path} ({changed} dòng thay đổi)")


# ====== CẤU HÌNH ======
md_files = [
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-001-mulapariyayasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-002-sabbasavasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-003-dhammadayadasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-004-bhayabheravasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-005-ananganasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-006-akankheyyasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-007-vatthasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-008-sallekhasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-009-sammaditthisutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-010-satipatthanasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-011-culasihanadasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-012-mahasihanadasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-013-mahadukkhakkhandhasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-014-culadukkhakkhandhasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-015-anumanasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-016-cetokhilasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-017-vanapatthasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-018-madhupindikasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-019-dvedhavitakkasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-020-vitakkasanthanasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-021-kakacupamasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-022-alagaddupamasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-023-vammikasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-024-rathavinitasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-025-nivapasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-026-pasarasisutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-027-culahatthipadopamasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-028-mahahatthipadopamasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-029-mahasaropamasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-030-culasaropamasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-031-culagosingasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-032-mahagosingasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-033-mahagopalakasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-034-culagopalakasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-035-culasaccakasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-036-mahasaccakasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-037-culatanhasankhayasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-038-mahatanhasankhayasutta copy.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-038-mahatanhasankhayasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-039-mahaassapurasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-040-culaassapurasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-041-saleyyakasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-042-veranjakasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-043-mahavedallasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-044-culavedallasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-045-culadhammasamadanasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-046-mahadhammasamadanasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-047-vimamsakasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-048-kosambiyasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-049-brahmanimantanikasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-050-maratajjaniyasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-051-kandarakasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-052-atthakanagarasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-053-sekhasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-054-potaliyasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-055-jivakasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-056-upalisutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-057-kukkuravatikasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-058-abhayarajakumarasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-059-bahuvedaniyasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-060-apannakasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-061-ambalatthikarahulovadasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-062-maharahulovadasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-063-culamalukyasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-064-mahamalukyasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-065-bhaddalisutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-066-latukikopamasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-067-catumasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-068-nalakapanasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-069-goliyanisutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-070-kitagirisutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-071-tevijjavacchasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-072-aggivacchasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-073-mahavacchasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-074-dighanakhasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-075-magandiyasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-076-sandakasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-077-mahasakuludayisutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-078-samanamundikasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-079-culasakuludayisutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-080-vekhanasasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-081-ghatikarasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-082-ratthapalasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-083-maghadevasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-084-madhurasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-085-bodhirajakumarasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-086-angulimalasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-087-piyajatikasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-088-bahitikasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-089-dhammacetiyasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-090-kannakatthalasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-091-brahmayusutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-092-selasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-093-assalayanasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-094-ghotamukhasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-095-cankisutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-096-esukarisutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-097-dhananjanisutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-098-vasetthasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-099-subhasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-100-sangaravasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-101-devadahasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-102-pancattayasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-103-kintisutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-104-samagamasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-105-sunakkhattasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-106-anenjasappayasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-107-ganakamoggallanasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-108-gopakamoggallanasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-109-mahapunnamasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-110-culapunnamasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-111-anupadasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-112-chabbisodhanasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-113-sappurisasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-114-sevitabbasevitabbasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-115-bahudhatukasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-116-isigilisutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-117-mahacattarisakasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-118-anapanassatisutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-119-kayagatasatisutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-120-sankharupapattisutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-121-culasunnatasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-122-mahasunnatasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-123-acchariyaabbhutasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-124-bakulasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-125-dantabhumisutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-126-bhumijasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-127-anuruddhasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-128-upakkilesasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-129-balapanditasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-130-devadutasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-131-bhaddekarattasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-132-anandabhaddekarattasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-133-mahakaccanabhaddekarattasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-134-lomasakangiyabhaddekarattasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-135-culakammavibhangasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-136-mahakammavibhangasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-137-salayatanavibhangasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-138-uddesavibhangasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-139-aranavibhangasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-140-dhatuvibhangasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-141-saccavibhangasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-142-dakkhinavibhangasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-143-anathapindikovadasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-144-channovadasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-145-punnovadasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-146-nandakovadasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-147-cularahulovadasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-148-chachakkasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-149-mahasalayatanikasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-150-nagaravindeyyasutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-151-pindapataparisuddhisutta.md",
"/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-152-indriyabhavanasutta.md",
  
    # Thêm các file khác...
]

# Đặt True để xem trước các thay đổi mà KHÔNG ghi file
DRY_RUN = False

for file in md_files:
    process_file(file, dry_run=DRY_RUN)

print("Hoàn tất!")

✅ Đã xử lý: /Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-001-mulapariyayasutta.md (47 dòng thay đổi)
✅ Đã xử lý: /Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-002-sabbasavasutta.md (22 dòng thay đổi)
✅ Đã xử lý: /Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-003-dhammadayadasutta.md (8 dòng thay đổi)
✅ Đã xử lý: /Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-004-bhayabheravasutta.md (28 dòng thay đổi)
✅ Đã xử lý: /Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-005-ananganasutta.md (25 dòng thay đổi)
✅ Đã xử lý: /Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-006-akankheyyasutta.md (20 dòng thay đổi)
✅ Đã xử lý: /Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-007-vatthasutta.md (16 dòng thay đổi)
✅ Đã xử lý: /Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-008-sallekhasutta.md (17 dòng thay đổi)
✅ Đã xử lý: /Users/ng/projects/nikaya2/docs/kinhtrungbo/pali/mn-009-sammaditthisutta.md (53 dòng thay đổi)
✅ Đã xử lý: /Users/ng/projects/nikaya2/docs/kinht

<>:15: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<>:15: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
/var/folders/0r/x3qsqbx96053svmhzm1mvb_00000gp/T/ipykernel_53624/446713250.py:15: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
  Thêm anchor {#number} vào cuối dòng nếu dòng bắt đầu bằng number\.
